# Model Evaluation and Operating Point Selection

This notebook evaluates the trained landslide model using a spatially held-out test split.
It reports:
- ROC-AUC
- PR-AUC
- recall/precision tradeoff
- validation threshold tuning
- final model operating point for deployment

The goal is to select a threshold that balances missed landslides against false alarms.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)

PROJECT_DIR = Path("..").resolve()
PROC_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "outputs" / "models"
METRIC_DIR = PROJECT_DIR / "outputs" / "metrics"
FIG_DIR = PROJECT_DIR / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cpu"
print("Project directories ready.")

In [ ]:
X = np.load(PROC_DIR / "X.npy").astype(np.float32)
y = np.load(PROC_DIR / "y.npy").astype(np.float32)
index = pd.read_csv(PROC_DIR / "dataset_index.csv")

val_mask = (index["split"] == "val").to_numpy()
test_mask = (index["split"] == "test").to_numpy()

print("X shape:", X.shape)
print("y positives:", int(y.sum()))
print("validation rows:", int(val_mask.sum()))
print("test rows:", int(test_mask.sum()))